# 02 Prompt Engineering：把人类先验变成可执行上下文

本节在第一个 Notebook 的真实数据划分和官方验证标准之上，比较六组逐步增强的 Prompt。实验固定模型为 `Qwen/Qwen3-4B-Thinking-2507`，任务固定为 Gravity、Equation Numeric 和 Bit Manipulation。

本 Notebook 的 DGX Spark 阶段只执行 CPU 构造和验证。模型加载、vLLM 服务和 GPU 推理都由明天的 RTX 5090 阶段执行，并且默认关闭。

## 学习目标

完成本节后，应该能够：

1. 说明 P0–P5 每一步新增的上下文组件；
2. 将课程提供的 Gravity、Equation Numeric 和 Bit Manipulation 先验写成不泄露当前答案的 Prompt；
3. 从 Train 集选择 Few-shot 示例，并确认没有读取 Validation / Holdout 的示例答案；
4. 使用统一的答案提取和验证逻辑，比较不同 Prompt 的结果；
5. 为明天的本地 vLLM 评测准备可复现的单实例推理接口。

## 1. 实验约束与逐步对照设计

P0 保持比赛原始 Prompt。P1–P5 是累积式设计：每一组只在上一组基础上增加一个明确组件。这样可以把变化归因到 Prompt 组件，而不是把所有提示词技巧混在一起。

| 版本 | 新增组件 | 作用 |
|---|---|---|
| P0 | Original | 原始比赛题面 |
| P1 | Output Contract | 固定最终答案格式 |
| P2 | Structured Reasoning | 明确解析、候选、验证、计算步骤 |
| P3 | Prior In-context | 注入课程确认的人类先验 |
| P4 | Few-shot | 从 Train 集加入 1–2 条示例 |
| P5 | Verify-before-answer | 输出前验证全部示例并处理不确定性 |

P4 的示例只来自 Train；当前题目的隐藏规则、标准答案和教师轨迹不会被写入 P3 或 P5 的规则说明。P4 中出现的参考答案只属于训练集演示样本，不能迁移成当前题目的运算映射。

In [ ]:
from __future__ import annotations

import ast
import json
import math
import os
import re
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "train.csv"
VALIDATION_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "validation.csv"
RESULT_DIR = PROJECT_ROOT / "data" / "processed" / "prompt_engineering"
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
VERIFIER_VERSION = "official_nvidia_metric_v2"
SEED = 42
RUN_GPU_EVAL = False
RUN_MODE = "smoke"                 # smoke / full
MAX_SAMPLES_PER_TASK = 3
MAX_NEW_TOKENS = 2048
VLLM_MAX_MODEL_LEN = int(os.environ.get("VLLM_MAX_MODEL_LEN", "8192"))
TEMPERATURE = 0.0
TOP_P = 1.0
PROMPT_VERSIONS = ("P0", "P1", "P2", "P3", "P4", "P5")
TARGET_TASKS = ("gravity", "equation_numeric", "bit_manipulation")

if not TRAIN_PATH.exists() or not VALIDATION_PATH.exists():
    raise FileNotFoundError("请先运行第一个 Notebook，准备 data/processed/splits/train.csv 和 validation.csv")

train_df = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False)
validation_df = pd.read_csv(VALIDATION_PATH, dtype=str, keep_default_na=False)
assert set(train_df["split"]) == {"train"}
assert set(validation_df["split"]) == {"validation"}
assert set(train_df["task_type"]) == set(TARGET_TASKS)
assert set(validation_df["task_type"]) == set(TARGET_TASKS)
assert not set(train_df["id"]) & set(validation_df["id"])

print({
    "model_id": MODEL_ID,
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "validation_by_task": validation_df["task_type"].value_counts().to_dict(),
    "run_gpu_eval": RUN_GPU_EVAL,
    "run_mode": RUN_MODE,
    "max_new_tokens": MAX_NEW_TOKENS,
})

## 2. 统一答案提取和验证

评测必须沿用第一个 Notebook 的标准：Gravity 和 Equation Numeric 使用 `rel_tol=1e-2, abs_tol=1e-5`，Bit Manipulation 使用 8 位字符串精确匹配。Prompt 版本不能改变验证方式，否则比较没有意义。

本 Notebook 在 Prompt 对照阶段采用 `strict_marker_first_prompt_eval`：只有 Boxed、Final/Answer 标签或独立答案行才算提取成功，不从未标记 Thinking 文本任意猜最后一个数字。该策略和版本号随每条 GPU 记录保存。


In [ ]:
def normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=re.I)
    return text.strip()


def extract_final_answer(text: str, task_type: str) -> tuple[str | None, str]:
    """Extract an explicitly marked final answer; never use an arbitrary CoT number."""
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return normalize_answer_text(boxed[-1]), "boxed"
    tagged = re.findall(r"<(?P<tag>final|answer)>\s*(?P<value>.*?)\s*</(?P=tag)>", text, flags=re.I | re.S)
    if tagged:
        return normalize_answer_text(tagged[-1][1]), "tagged"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        labeled = re.match(r"(?i)^(?:final answer|answer|最终答案|最终输出)\s*[:：]\s*(.+)$", line)
        if labeled:
            candidate = normalize_answer_text(labeled.group(1)).strip("`$")
            if task_type == "bit_manipulation" and re.fullmatch(r"(?:0b)?[01]{8}", candidate):
                return candidate.removeprefix("0b"), "final_label"
            if task_type in {"gravity", "equation_numeric"} and re.fullmatch(r"[-+]?\d+(?:\.\d+)?", candidate):
                return candidate, "final_label"
        if task_type == "bit_manipulation" and re.fullmatch(r"(?:0b)?[01]{8}", line):
            return line.removeprefix("0b"), "standalone_final_line"
        if task_type in {"gravity", "equation_numeric"} and re.fullmatch(r"[-+]?\d+(?:\.\d+)?", line):
            return line, "standalone_final_line"
    return None, "unmarked_or_empty"

OFFICIAL_NUMERIC_REL_TOL = 1e-2
OFFICIAL_NUMERIC_ABS_TOL = 1e-5


def official_numeric_match(predicted: object, expected: object) -> bool:
    try:
        predicted_value = float(normalize_answer_text(predicted))
        expected_value = float(normalize_answer_text(expected))
    except (TypeError, ValueError):
        return False
    return (
        math.isfinite(predicted_value)
        and math.isfinite(expected_value)
        and math.isclose(
            predicted_value,
            expected_value,
            rel_tol=OFFICIAL_NUMERIC_REL_TOL,
            abs_tol=OFFICIAL_NUMERIC_ABS_TOL,
        )
    )


def verify_answer(task_type: str, predicted: object, expected: object) -> tuple[bool, str]:
    if predicted is None:
        return False, "not_extracted"
    predicted_text = normalize_answer_text(predicted)
    expected_text = normalize_answer_text(expected)
    if task_type == "bit_manipulation":
        p, e = predicted_text.removeprefix("0b"), expected_text.removeprefix("0b")
        if not re.fullmatch(r"[01]{8}", p) or not re.fullmatch(r"[01]{8}", e):
            return False, "invalid_8bit_format"
        return p == e, "exact_8bit"
    if task_type in {"gravity", "equation_numeric"}:
        if not re.fullmatch(r"[-+]?\d+(?:\.\d+)?", predicted_text):
            return False, "invalid_numeric_format"
        return official_numeric_match(predicted_text, expected_text), "official_math_isclose"
    return predicted_text == expected_text, "exact_text"

## 3. 课程先验：只描述规则空间，不给出当前题目的隐藏规则

P3 才加入下面的先验。它们来自第一个 Notebook 已确认的课程定义。先验文本包含候选规则和全局约束，但不包含任何验证集样本的 `answer`、隐藏 `g`、隐藏 Equation 运算符分配或隐藏 Bit 表达式。

In [ ]:
# 24 条 Equation Numeric 规则，保持与第一个 Notebook 的 Prior Specification 一致。
NUMERIC24_RULES = [
    {"name": "sub", "formula": "AB - CD, raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "absdiff", "formula": "|AB - CD|, raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "absdiff_prefix", "formula": "|AB - CD| with operator prefix", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "join", "formula": "AB concatenated with CD", "family": "join", "group": "join_like"},
    {"name": "join_swap", "formula": "CD concatenated with AB", "family": "join", "group": "join_like"},
    {"name": "add", "formula": "AB + CD", "family": "normal", "group": "add_like"},
    {"name": "add_p1", "formula": "AB + CD + 1", "family": "normal", "group": "add_like"},
    {"name": "add_m1", "formula": "AB + CD - 1", "family": "normal", "group": "add_like"},
    {"name": "mul", "formula": "AB * CD", "family": "normal", "group": "mul_like"},
    {"name": "mul_p1", "formula": "AB * CD + 1", "family": "normal", "group": "mul_like"},
    {"name": "mul_m1", "formula": "AB * CD - 1", "family": "normal", "group": "mul_like"},
    {"name": "maxmodmin", "formula": "max(AB, CD) mod min(AB, CD)", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "flip_add", "formula": "reverse(BA + DC)", "family": "flip", "group": "add_like"},
    {"name": "flip_add_p1", "formula": "reverse(BA + DC + 1)", "family": "flip", "group": "add_like"},
    {"name": "flip_add_m1", "formula": "reverse(BA + DC - 1)", "family": "flip", "group": "add_like"},
    {"name": "flip_mul", "formula": "reverse(BA * DC)", "family": "flip", "group": "mul_like"},
    {"name": "flip_mul_p1", "formula": "reverse(BA * DC + 1)", "family": "flip", "group": "mul_like"},
    {"name": "flip_mul_m1", "formula": "reverse(BA * DC - 1)", "family": "flip", "group": "mul_like"},
    {"name": "flip_absdiff", "formula": "reverse(|BA - DC|)", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_maxmodmin", "formula": "reverse(max(BA, DC) mod min(BA, DC))", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_sub_prefix", "formula": "reverse(BA - DC), preserving a negative sign as an operator prefix", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_sub_suffix", "formula": "reverse(DC - BA), appending the operator for a negative result", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_absdiff_prefix", "formula": "operator-prefixed reverse(|BA - DC|)", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_absdiff_suffix", "formula": "reverse(|BA - DC|) followed by the operator", "family": "flip", "group": "sub_or_mod_like"},
]

EQUATION_PRIOR_TEXT = """Equation Numeric prior (candidate rules, not the current hidden assignment):
- Each expression has exactly two decimal-digit operands: AB<operator>CD.
- AB_CD parses left=AB, right=CD. BA_DC parses left=reverse(AB), right=reverse(CD).
- Candidate arithmetic/output rules are:
""" + "\n".join(
    f"  - {rule['name']}: {rule['formula']}" for rule in NUMERIC24_RULES
) + """
- Global constraints:
  1. All non-join rules in one prompt use either the normal or flipped operand family.
  2. At most one non-join rule is selected from each group: sub_or_mod_like, add_like, mul_like.
  3. join and join_swap are join_like exceptions and may coexist with other groups.
  4. If the query operator is absent from examples, classify it as Guess. Keep the answer only when all valid assignments produce one query output; otherwise report ambiguity.
- Always distinguish Deduce (query operator appeared in examples) from Guess (query operator did not appear).
"""

GRAVITY_PRIOR_TEXT = """Gravity prior:
- Use distance = 0.5 * g * t^2, with t in seconds, distance in meters, and g in m/s^2.
- Estimate the current question's g independently from its own public examples.
- Keep the observed numeric precision and use the official numeric tolerance.
- Do not assume g=9.8 and do not reuse g inferred from another question.
"""

BIT_PRIOR_TEXT = """Bit Manipulation prior:
- Values are unsigned 8-bit bytes; every result is masked to 0..255 and printed as 8 bits.
- Legal expressions use x, NOT, SHL, SHR, ROTL, ROTR, AND, OR, XOR.
- SHL/SHR shift by k in 0..7; ROTL/ROTR rotate within 8 bits.
- Grammar: expr := x | NOT(expr) | SHL(expr,k) | SHR(expr,k) | ROTL(expr,k) | ROTR(expr,k) | AND(expr,expr) | OR(expr,expr) | XOR(expr,expr).
- Infer the hidden expression from all input-output examples; the expression itself is not supplied.
"""

TASK_PRIOR_TEXT = {
    "gravity": GRAVITY_PRIOR_TEXT,
    "equation_numeric": EQUATION_PRIOR_TEXT,
    "bit_manipulation": BIT_PRIOR_TEXT,
}

assert len(NUMERIC24_RULES) == 24
assert all("ground truth" not in text.lower() and "hidden_rule" not in text.lower() for text in TASK_PRIOR_TEXT.values())
print({"equation_numeric_rules": len(NUMERIC24_RULES), "prior_tasks": list(TASK_PRIOR_TEXT)})

## 4. P0–P5 Prompt 构造

构造函数只读取目标行的 `id/task_type/prompt` 和训练集 Few-shot 候选；它不读取目标行的 `answer`。因此 Prompt 中的 `answer` 字段不会被误当成人类先验。每个版本返回 Prompt 文本和审计元数据，方便明天保存每条结果。

In [ ]:
PROMPT_COMPONENTS = {
    "P0": (),
    "P1": ("output_contract",),
    "P2": ("output_contract", "structured_reasoning"),
    "P3": ("output_contract", "structured_reasoning", "human_prior"),
    "P4": ("output_contract", "structured_reasoning", "human_prior", "train_few_shot"),
    "P5": ("output_contract", "structured_reasoning", "human_prior", "train_few_shot", "verify_before_answer"),
}

OUTPUT_CONTRACT_TEXT = """Output contract:
- Keep the reasoning concise and auditable.
- Put exactly one final answer in \\boxed{...}.
- Gravity: use a decimal numeric answer.
- Equation Numeric: preserve an integer, sign, or digit-concatenation format exactly.
- Bit Manipulation: output exactly eight binary digits.
"""

STRUCTURED_REASONING_TEXT = """Structured reasoning procedure:
1. Parse every example and the Query without skipping rows.
2. List candidate rules or expressions supported by the examples.
3. Check each candidate against all examples and apply the task's global constraints.
4. Decide whether the Query is Deduce or Guess and whether its answer is unique.
5. Compute the Query only after the previous checks; then follow the output contract.
"""

VERIFY_BEFORE_ANSWER_TEXT = """Verify-before-answer:
- Recalculate every displayed example with the selected candidate before answering.
- If multiple candidates remain but produce different Query outputs, state that the answer is ambiguous and do not invent a deterministic answer.
- If candidates are semantically different but agree on this Query, mark the answer as answer-equivalent.
- Do not transfer a hidden rule, parameter, or operator mapping from a Few-shot example to the current question.
"""


def choose_train_few_shot(task_type: str, count: int = 2) -> list[dict[str, str]]:
    """Deterministic seeded Train-only sample; never choose by smallest ID."""
    pool = train_df[train_df["task_type"] == task_type].copy()
    task_seed = {"gravity": 1, "equation_numeric": 2, "bit_manipulation": 3}[task_type]
    selected = pool.sample(n=min(int(count), len(pool)), random_state=SEED + task_seed)
    return [
        {"id": str(row.id), "task_type": str(row.task_type), "prompt": str(row.prompt), "answer": str(row.answer), "source_split": "train"}
        for row in selected.itertuples(index=False)
    ]

def _few_shot_text(task_type: str) -> tuple[str, list[str]]:
    examples = choose_train_few_shot(task_type, count=2)
    blocks = []
    ids = []
    for index, example in enumerate(examples, start=1):
        ids.append(example["id"])
        blocks.append(
            f"### Train demonstration {index} (task={task_type})\n"
            f"Problem:\n{example['prompt']}\n"
            f"Reference final answer: {example['answer']}\n"
        )
    text = (
        "Few-shot note: demonstrations are independent questions. Their hidden rule or parameter does not transfer to the current question.\n"
        + "\n".join(blocks)
    )
    return text, ids


def build_prompt_variants(row: pd.Series) -> tuple[dict[str, str], dict[str, object]]:
    """Build cumulative P0-P5 prompts without reading the target answer."""
    task_type = str(row["task_type"])
    original_prompt = str(row["prompt"])
    assert task_type in TARGET_TASKS
    p0 = original_prompt
    p1 = p0 + "\n\n" + OUTPUT_CONTRACT_TEXT
    p2 = p1 + "\n" + STRUCTURED_REASONING_TEXT
    p3 = p2 + "\n" + TASK_PRIOR_TEXT[task_type]
    few_shot_text, few_shot_ids = _few_shot_text(task_type)
    p4 = p3 + "\n" + few_shot_text
    p5 = p4 + "\n" + VERIFY_BEFORE_ANSWER_TEXT
    variants = {"P0": p0, "P1": p1, "P2": p2, "P3": p3, "P4": p4, "P5": p5}
    metadata = {
        "sample_id": str(row["id"]),
        "task_type": task_type,
        "few_shot_source_split": "train",
        "few_shot_ids": few_shot_ids,
        "target_answer_injected": False,
        "target_hidden_rule_injected": False,
        "components": PROMPT_COMPONENTS,
    }
    return variants, metadata

### 三类任务的先验边界

- **Gravity**：只给公式、参数含义、单位和精度；当前题目的 `g` 必须由当前题面示例估计。
- **Equation Numeric**：P3 给出 24 条规则的数学定义、操作数解析、格式规则和全局约束；不提供当前运算符到规则的分配。
- **Bit Manipulation**：P3 给出 8-bit DSL 和合法语法；不提供当前题目的表达式。
- **Few-shot**：P4 的演示题来自 Train，并显式提醒每道题规则独立。

In [ ]:
# 展示三个任务的六组 Prompt 元数据；这里只打印长度和组件，避免输出整段长 Prompt。
representative_rows = validation_df.sort_values(["task_type", "id"]).groupby("task_type", sort=True).head(1)
representative_audit = []
for row in representative_rows.itertuples(index=False):
    variants, metadata = build_prompt_variants(pd.Series(row._asdict()))
    representative_audit.append({
        "task_type": row.task_type,
        "sample_id": row.id,
        "versions": list(variants),
        "lengths": {version: len(text) for version, text in variants.items()},
        "components": metadata["components"],
        "few_shot_ids": metadata["few_shot_ids"],
    })
print(json.dumps(representative_audit, ensure_ascii=False, indent=2))

## 5. CPU Prompt 审计：验证组件递增、先验隔离和 Few-shot 来源

这里不调用模型，只构造完整 Validation 的 393 × 6 个 Prompt。审计重点是：P0 是否保持原文、P1–P5 是否逐步增加组件、P3 是否只包含公开先验、P4 是否只使用 Train 示例，以及所有版本是否作用于完全相同的 Validation 样本。

In [ ]:
def audit_prompt_variants(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    rows, metadata_rows = [], []
    for row in frame.to_dict("records"):
        variants, metadata = build_prompt_variants(pd.Series(row))
        assert variants["P0"] == row["prompt"]
        assert metadata["target_answer_injected"] is False
        assert metadata["target_hidden_rule_injected"] is False
        assert metadata["few_shot_source_split"] == "train"
        assert set(metadata["few_shot_ids"]).issubset(set(train_df["id"]))
        assert not set(metadata["few_shot_ids"]) & set(validation_df["id"])
        for version in PROMPT_VERSIONS:
            rows.append({
                "sample_id": row["id"],
                "task_type": row["task_type"],
                "prompt_version": version,
                "prompt": variants[version],
                "prompt_length_chars": len(variants[version]),
                "few_shot_ids": metadata["few_shot_ids"],
            })
        metadata_rows.append({"sample_id": row["id"], **metadata})
    result = pd.DataFrame(rows)
    assert len(result) == len(frame) * len(PROMPT_VERSIONS)
    assert result.groupby("prompt_version")["sample_id"].nunique().eq(len(frame)).all()
    assert result.groupby("prompt_version")["task_type"].value_counts().groupby(level=0).sum().eq(len(frame)).all()
    return result, metadata_rows


validation_prompt_df, validation_prompt_metadata = audit_prompt_variants(validation_df)
print({
    "validation_samples": int(validation_df["id"].nunique()),
    "constructed_prompts": int(len(validation_prompt_df)),
    "prompt_versions": validation_prompt_df["prompt_version"].unique().tolist(),
    "by_version": validation_prompt_df.groupby("prompt_version").size().to_dict(),
    "few_shot_source_splits": sorted({row["few_shot_source_split"] for row in validation_prompt_metadata}),
})

## 6. 明天的 RTX 5090 vLLM 接口（当前不执行）

函数参考第一个 Notebook 的实现，但把 `LLM(...)` 放在评测函数内部且只实例化一次。P0–P5 共用该实例、同一套解码参数和同一批 Validation 样本。`RUN_GPU_EVAL=False` 是硬性安全默认值；本阶段不会导入 vLLM、Tokenizer 或模型。

### 6.1 Qwen3 Chat Template 渲染

In [ ]:
def render_qwen3_prompt(tokenizer, original_prompt: str) -> str:
    """Use Qwen3's official chat template with thinking enabled."""
    messages = [{"role": "user", "content": original_prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except TypeError:
        # Compatibility path for an older Transformers template; no manual special tokens.
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


### 6.2 Prompt 结果汇总：分任务指标与成本

In [ ]:
def summarize_prompt_records(records: list[dict[str, object]]) -> dict[str, object]:
    frame = pd.DataFrame(records)
    if frame.empty:
        return {"records": 0, "status": "no_records"}
    by_prompt_task = {}
    for (prompt_version, task_type), group in frame.groupby(["prompt_version", "task_type"], sort=True):
        by_prompt_task[f"{prompt_version}/{task_type}"] = {
            "records": int(len(group)),
            "accuracy": float(group["correct"].mean()),
            "answer_extraction_success_rate": float(group["extraction_success"].mean()),
            "invalid_output_rate": float((~group["extraction_success"]).mean()),
            "truncation_rate": float(group["truncated"].mean()),
            "average_generation_tokens": float(group["generation_tokens"].mean()),
            "average_inference_time_seconds": float(group["inference_time_seconds"].mean()),
            "total_inference_time_seconds": float(group["inference_time_seconds"].sum()),
        }
    overall = frame.groupby("prompt_version")["correct"].mean().to_dict()
    macro = frame.groupby(["prompt_version", "task_type"])["correct"].mean().groupby(level=0).mean().to_dict()
    return {
        "records": int(len(frame)),
        "by_prompt_task": by_prompt_task,
        "overall_accuracy_sample_weighted": {key: float(value) for key, value in overall.items()},
        "macro_accuracy": {key: float(value) for key, value in macro.items()},
    }


### 6.3 P0–P5 统一 vLLM 评测

模型只加载一次，所有 Prompt 版本复用同一 Validation 样本和解码预算。

In [ ]:
def run_gpu_evaluation(
    evaluation_frame: pd.DataFrame,
    model_id: str = MODEL_ID,
    output_dir: Path = RESULT_DIR,
) -> dict[str, object]:
    """Run P0-P5 on the same Validation rows with one vLLM instance."""
    if not RUN_GPU_EVAL:
        return {"status": "skipped_by_default", "reason": "RUN_GPU_EVAL=False；DGX Spark 阶段不加载模型、不启动 vLLM、不执行 GPU 推理。"}
    if set(evaluation_frame["split"].unique()) != {"validation"}:
        raise ValueError("Prompt Engineering 只能在 Validation 上选择和比较 Prompt")
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    model_source = os.environ.get("LOCAL_MODEL_PATH", model_id)
    local_files_only = Path(model_source).exists()
    tokenizer = AutoTokenizer.from_pretrained(model_source, local_files_only=local_files_only, trust_remote_code=True)
    max_model_len = VLLM_MAX_MODEL_LEN
    llm = LLM(model=model_source, tokenizer=model_source, tensor_parallel_size=1, gpu_memory_utilization=float(os.environ.get("VLLM_GPU_MEMORY_UTILIZATION", "0.85")), max_model_len=max_model_len, enforce_eager=True)
    sampling_params = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS)
    prompt_rows = []
    for row in evaluation_frame.to_dict("records"):
        variants, metadata = build_prompt_variants(pd.Series(row))
        for prompt_version in PROMPT_VERSIONS:
            prompt_rows.append({"sample_id": str(row["id"]), "task_type": str(row["task_type"]), "prompt_version": prompt_version, "prompt": variants[prompt_version], "expected_answer": str(row["answer"]), "few_shot_ids": metadata["few_shot_ids"]})
    rendered_by_version = {}
    for version in PROMPT_VERSIONS:
        batch = [row for row in prompt_rows if row["prompt_version"] == version]
        rendered = [render_qwen3_prompt(tokenizer, row["prompt"]) for row in batch]
        tokenized = tokenizer(rendered, add_special_tokens=False)["input_ids"]
        overflow = [len(ids) + MAX_NEW_TOKENS > max_model_len for ids in tokenized]
        if any(overflow):
            raise ValueError(f"{version} context preflight failed: {sum(overflow)} rows exceed VLLM_MAX_MODEL_LEN")
        rendered_by_version[version] = (batch, rendered, tokenized)
    # Warmup is outside measured wall time and shared by all prompt versions.
    llm.generate([rendered_by_version["P0"][1][0]], sampling_params, use_tqdm=False)
    records = []
    batch_metrics = {}
    for prompt_version in PROMPT_VERSIONS:
        batch, rendered_prompts, tokenized = rendered_by_version[prompt_version]
        started = time.perf_counter()
        outputs = llm.generate(rendered_prompts, sampling_params, use_tqdm=True)
        batch_wall = time.perf_counter() - started
        batch_metrics[prompt_version] = {"batch_wall_time_seconds": batch_wall, "batch_size": len(outputs), "batch_output_tokens": int(sum(len(getattr(request.outputs[0], "token_ids", ()) or ()) for request in outputs))}
        for row, request_output, prompt_ids in zip(batch, outputs, tokenized):
            generated = request_output.outputs[0]
            raw_output = generated.text
            extracted, extraction_method = extract_final_answer(raw_output, row["task_type"])
            correct, verification_method = verify_answer(row["task_type"], extracted, row["expected_answer"])
            token_ids = list(getattr(generated, "token_ids", ()) or ())
            finish_reason = getattr(generated, "finish_reason", None)
            records.append({**row, "raw_output": raw_output, "extracted_answer": extracted, "extraction_method": extraction_method, "verification_method": verification_method, "verifier_version": VERIFIER_VERSION, "extraction_success": extracted is not None, "correct": bool(correct), "prompt_tokens": len(prompt_ids), "completion_tokens": len(token_ids), "generation_tokens": len(token_ids), "finish_reason": finish_reason, "truncated": bool(finish_reason == "length" or len(token_ids) >= MAX_NEW_TOKENS), "batch_wall_time_seconds": batch_wall, "amortized_inference_time_seconds": batch_wall / max(len(outputs), 1), "inference_time_seconds": batch_wall / max(len(outputs), 1), "token_throughput": len(token_ids) / max(batch_wall, 1e-9)})
    output_dir.mkdir(parents=True, exist_ok=True)
    result_path = output_dir / f"prompt_engineering_{RUN_MODE}_results.jsonl"
    result_path.write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in records) + "\n", encoding="utf-8")
    summary = summarize_prompt_records(records)
    summary.update({"model_id": model_id, "model_source": model_source, "selection_split": "validation", "run_mode": RUN_MODE, "max_new_tokens": MAX_NEW_TOKENS, "max_model_len": max_model_len, "temperature": TEMPERATURE, "top_p": TOP_P, "prompt_versions": list(PROMPT_VERSIONS), "batch_metrics": batch_metrics, "warmup": "one P0 request excluded from timings", "result_path": str(result_path)})
    (output_dir / f"prompt_engineering_{RUN_MODE}_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    del llm
    try:
        import gc
        gc.collect()
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception:
        pass
    return {"status": "completed", "summary": summary, "results_path": str(result_path)}


### 6.4 GPU 评测执行入口

In [ ]:
if RUN_GPU_EVAL:
    evaluation_frame = validation_df.copy()
    if RUN_MODE == "smoke":
        evaluation_frame = pd.concat(
            [group.head(MAX_SAMPLES_PER_TASK) for _, group in validation_df.groupby("task_type", sort=True)],
            ignore_index=True,
        )
    gpu_status = run_gpu_evaluation(evaluation_frame)
else:
    gpu_status = {
        "status": "skipped_by_default",
        "reason": "DGX Spark 阶段关闭 GPU 评测。明天在 RTX 5090 上先运行 smoke，再运行 full。",
    }
print(json.dumps(gpu_status, ensure_ascii=False, indent=2))


## 7. 结果分析接口

GPU 运行后，`summarize_prompt_records` 会对每个 Prompt × 任务单独统计 Accuracy、答案提取成功率、Invalid Output、截断率、平均生成 Token、平均耗时和总耗时，同时给出样本加权 Overall Accuracy 与任务宏平均 Macro Accuracy。Holdout 不参与 Prompt 选择；只有最终确定方案后，才可以在后续实验中另行报告 Holdout。

In [ ]:
def collect_error_cases(records: list[dict[str, object]], prompt_version: str | None = None, limit: int = 5) -> list[dict[str, object]]:
    """Return inspectable failures without changing any dataset file."""
    selected = [row for row in records if prompt_version is None or row.get("prompt_version") == prompt_version]
    failures = [row for row in selected if not row.get("correct", False)]
    return [
        {
            "sample_id": row.get("sample_id"),
            "task_type": row.get("task_type"),
            "prompt_version": row.get("prompt_version"),
            "expected_answer": row.get("expected_answer"),
            "extracted_answer": row.get("extracted_answer"),
            "finish_reason": row.get("finish_reason"),
            "truncated": row.get("truncated"),
            "extraction_method": row.get("extraction_method"),
            "raw_output": row.get("raw_output", "")[:1000],
        }
        for row in failures[:limit]
    ]


print("GPU 结果尚未生成；运行后可用 collect_error_cases(records, prompt_version='P3') 检查错误样本。")

## 8. DGX Spark CPU 验收

下面的测试不加载模型、不启动 vLLM，也不读取 Holdout。它覆盖 Prompt 构造、P0–P5 组件递增、Train-only Few-shot、先验与目标答案隔离、统一答案提取和三类答案验证。

In [ ]:
# 1) 每个版本都覆盖同一批 Validation 样本，且 P0 保持原题面。
assert len(validation_prompt_df) == len(validation_df) * 6
assert set(validation_prompt_df["prompt_version"]) == set(PROMPT_VERSIONS)
assert validation_prompt_df.groupby("prompt_version")["sample_id"].nunique().eq(len(validation_df)).all()

# 2) 组件只能逐步增加，不能在早期版本泄露完整先验。
for row in validation_df.head(12).to_dict("records"):
    variants, metadata = build_prompt_variants(pd.Series(row))
    assert variants["P0"] == row["prompt"]
    assert len(variants["P0"]) < len(variants["P1"]) < len(variants["P2"]) < len(variants["P3"])
    assert len(variants["P3"]) < len(variants["P4"]) < len(variants["P5"])
    assert TASK_PRIOR_TEXT[row["task_type"]] not in variants["P0"]
    assert TASK_PRIOR_TEXT[row["task_type"]] in variants["P3"]
    assert metadata["target_answer_injected"] is False
    assert metadata["target_hidden_rule_injected"] is False
    assert set(metadata["few_shot_ids"]).issubset(set(train_df["id"]))

# 3) Few-shot 只能来自 Train，不能跨题迁移隐藏规则。
assert all(item["few_shot_source_split"] == "train" for item in validation_prompt_metadata)
assert all(set(item["few_shot_ids"]).issubset(set(train_df["id"])) for item in validation_prompt_metadata)
assert not set(validation_prompt_df["few_shot_ids"].explode()) & set(validation_df["id"])

# 4) 三类统一答案提取和验证的正、负样本。
assert extract_final_answer(r"推理\n\boxed{12.34}", "gravity")[0] == "12.34"
assert verify_answer("gravity", "12.34", "12.34")[0]
assert verify_answer("gravity", "12.34001", "12.34")[0]
assert extract_final_answer(r"答案：\boxed{-42}", "equation_numeric")[0] == "-42"
assert verify_answer("equation_numeric", "-42", "-42")[0]
assert extract_final_answer(r"最终输出 \boxed{01010101}", "bit_manipulation")[0] == "01010101"
assert verify_answer("bit_manipulation", "01010101", "01010101")[0]
assert not verify_answer("bit_manipulation", "01010100", "01010101")[0]
assert not verify_answer("gravity", "not-a-number", "1.0")[0]
assert extract_final_answer("step 1: 12\nstep 2: 34", "gravity")[0] is None

# 5) GPU 入口存在但保持关闭；本单元不会触发 vLLM。
assert RUN_GPU_EVAL is False
assert RUN_MODE == "smoke"
assert MAX_NEW_TOKENS == 2048
assert "vllm" not in sys.modules

cpu_test_report = {
    "status": "passed",
    "validation_samples": int(len(validation_df)),
    "prompt_records": int(len(validation_prompt_df)),
    "prompt_versions": list(PROMPT_VERSIONS),
    "few_shot_source": "train_only",
    "prior_hidden_rule_isolation": True,
    "answer_extraction_and_verifier": "passed",
    "verifier_version": VERIFIER_VERSION,
    "extraction_policy": "strict_marker_first_prompt_eval",
    "gpu_eval": "disabled",
    "run_mode": RUN_MODE,
    "max_new_tokens": MAX_NEW_TOKENS,
}
(RESULT_DIR / "prompt_cpu_smoke_report.json").parent.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / "prompt_cpu_smoke_report.json").write_text(json.dumps(cpu_test_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(cpu_test_report, ensure_ascii=False, indent=2))

## 本节总结与明天的测试步骤

今天完成了六组可审计 Prompt、三类任务的公开先验边界、Train-only Few-shot 选择、统一答案提取/验证和单实例 vLLM 评测接口。CPU 测试只证明 Prompt 构造与评测代码的结构正确，不代表模型效果。

明天在 RTX 5090 上：

1. 检查 GPU 是否空闲，不终止其他进程；
2. 在现有 vLLM 环境加载一次 `Qwen/Qwen3-4B-Thinking-2507`；
3. `RUN_GPU_EVAL=True, RUN_MODE="smoke"`，每类 5 条运行 P0–P5；
4. 检查 `finish_reason`、`truncated`、显存和生成吞吐；
5. Smoke 正常后运行完整 Validation；
6. 只使用 Validation 选择最佳 Prompt，保留逐条 JSONL 和汇总指标；
7. 不用 Holdout 结果修改 Prompt，也不把截断回答当作正常完成。

后续思考：如果 P3 提升了规则识别但 P5 降低了输出截断率，应该分别报告能力变化和输出格式变化；不能只看一个总 Accuracy。